# Tic Tac Toe with Competing Agents

## Scenario

This experiment is distinct from the previous 'competing drive' notebooks.

This time I wanted to attempt to get an agent to play a game of Tic Tac Toe without any explicitly encoded rules, purely through preference alignment.

Furthermore, I wanted it to play against a copy of itself.

Being a game of perfect information this seemed like a reasonable challenge, with a view towards modelling partial information games like poker in the future.

I was successful, with some caveats:
1. It works perfectly on a 3 * 3 grid. Moving to 4*4 I see some non-optimal moves but I think that parameter tuning could fix this.
2. It takes a couple of seconds to run a full 3 * 3 grid game, but a 4 * 4 takes 5 minutes **per move** due to moving from 3^9 to 3^16 possible cell state combinations.

Parameter tuning is slow because I have to use a 4 * 4 grid to evaluate changes. If I can speed up the inference, I can experiment with different settings quickly.

In the next notebook I plan to explore the JAX version of pymdp to see if it is faster.

In [20]:
%pip install inferactively-pymdp

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import numpy as np
import matplotlib.pyplot as plt
from pymdp.agent import Agent
from pymdp import utils
from copy import deepcopy

## Hidden States

We will begin by defining a few constants to make the following code easier to read.

Each cell can be one of Empty, O, or X.

It is either Player O's or Player X's turn.

We also track the empty count, or 'moves left' as a distinct cell state.

> We could infer an observation of empty count from the cell states, but we need it in the state itself because in this implementation it is used for the B matrix transition during the opponent's turn to calculate the chance of the remaining cells being taken. More info in the B section below.

In [22]:
grid_size = 3
win_length = 3  # e.g. 3 for classic tic tac toe
n_cells = grid_size * grid_size

# Empty Count options (0 to n_cells inclusive,so 17 vals for 16 cells)
empty_count_states = list(range(0, n_cells + 1))

CELL_EMPTY = 0
CELL_O = 1
CELL_X = 2
n_cell_states = 3

PLAYER_O = 0
PLAYER_X = 1
n_turn_states = 2  # O's turn, X's turn

ACTION_STAY = 0
ACTION_MARK = 1
n_empty_count_states = len(empty_count_states)

n_states = [n_cell_states] * n_cells + [n_turn_states] + [n_empty_count_states]

print(f"Number of states\n{n_cell_states} cell states * {n_cells} cells, 2 turn states, {n_empty_count_states} empty states:\n{num_states}")

Number of states
3 cell states * 9 cells, 2 turn states, 10 empty states:
[3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 10]


## A Matrix: Observation -> State

The agent makes confident observations of each hidden state factor (cells, turn and empty count).

The solution I have come up with also *infers* an observation of each possible cell combination of minimum `win_length`, which I will refer to as 'chunks'.

Chunks can be observed to be
- Empty
- Unwinnable (a mix of players)
- OIn`n` (number of moves until O wins, from 0 (won) to win_lenth-1 (one cell taken))
- XIn`n` (as above)

Later on we will see how the agent's preferences are represented in terms of observed chunks.

In [23]:
chunk_obs_labels = ['Empty', 'Unwinnable'] + [f'OIn{i}' for i in range(0, win_length)] + [f'XIn{i}' for i in range(0, win_length)]

def get_chunks(grid_size, win_length):
    """Return a list of lists, each sublist is indices of a winning line."""
    chunks = []
    # Rows
    for r in range(grid_size):
        for c in range(grid_size - win_length + 1):
            chunks.append([r * grid_size + c + i for i in range(win_length)])
    # Columns
    for c in range(grid_size):
        for r in range(grid_size - win_length + 1):
            chunks.append([(r + i) * grid_size + c for i in range(win_length)])
    # Diagonals
    for r in range(grid_size - win_length + 1):
        for c in range(grid_size - win_length + 1):
            # Down-right
            chunks.append([(r + i) * grid_size + (c + i) for i in range(win_length)])
            # Down-left
            chunks.append([(r + i) * grid_size + (c + win_length - 1 - i) for i in range(win_length)])
    return chunks

chunks = get_chunks(grid_size, win_length)
n_chunks = len(chunks)

cell_obs = [ 3 for _ in range(n_cells) ]
chunk_obs = [ len(chunk_obs_labels) for _ in range(n_chunks) ]
n_obs = [*cell_obs, 2, len(empty_count_states), *chunk_obs]

print("Chunk labels:", chunk_obs_labels)
print(f"Number of Chunks: {n_chunks}")
print(f"Chunk cell indices: {chunks}")
print(f"Observation Dimensionalities: {n_obs}")

Chunk labels: ['Empty', 'Unwinnable', 'OIn0', 'OIn1', 'OIn2', 'XIn0', 'XIn1', 'XIn2']
Number of Chunks: 8
Chunk cell indices: [[0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]]
Observation Dimensionalities: [3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 10, 8, 8, 8, 8, 8, 8, 8, 8]


In order to attempt to fight the impending combinatorial explosion, this time around I have restricted the A matrix mappings to only depend on the factors which affect them, just as we have done with the B matrix previously.

- Each cell observation depends only on the hidden state of the cell it represents.
- The turn and empty count observations also only depend on the matching hidden states.
- The chunk observations each depend only on the cells which define them. For example with win length 3, chunk 1 would depend on cells 0, 1 and 2.

In [24]:
A_factor_list = []

for i in range(n_cells):
    A_factor_list.append([i])  # cell obs depend on cell state

A_factor_list.append([n_cells])  # turn obs depends on turn state
A_factor_list.append([n_cells+1])  # empty count obs depends on empty count state

for i, chunk in enumerate(chunks): # chunks depend on the cells they observe
    A_factor_list.append(chunk)

print(f"A dependencies: {A_factor_list}")

A dependencies: [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]]


We'll initialise the A matrix with zeroes. It has one state for each cell obs, turn obs, and empty count obs plus win_length states for each chunk obs

In [25]:
A = utils.obj_array(len(n_obs))
for i in range(len(n_obs)):
    A[i] = np.zeros((n_obs[i],) + tuple([n_states[j] for j in A_factor_list[i]]))

# For each modality, its observation count followed by the counts of every hidden state it depends on.
for i in range(len(n_obs)):
    print(f"A[{i}] obs shape: {A[i].shape}")

A[0] obs shape: (3, 3)
A[1] obs shape: (3, 3)
A[2] obs shape: (3, 3)
A[3] obs shape: (3, 3)
A[4] obs shape: (3, 3)
A[5] obs shape: (3, 3)
A[6] obs shape: (3, 3)
A[7] obs shape: (3, 3)
A[8] obs shape: (3, 3)
A[9] obs shape: (2, 2)
A[10] obs shape: (10, 10)
A[11] obs shape: (8, 3, 3, 3)
A[12] obs shape: (8, 3, 3, 3)
A[13] obs shape: (8, 3, 3, 3)
A[14] obs shape: (8, 3, 3, 3)
A[15] obs shape: (8, 3, 3, 3)
A[16] obs shape: (8, 3, 3, 3)
A[17] obs shape: (8, 3, 3, 3)
A[18] obs shape: (8, 3, 3, 3)


We need a function which can convert a list of cell states into a chunk observation

In [26]:
def get_chunk_obs(chunk_cell_states):
    n_empty = sum([s == CELL_EMPTY for s in chunk_cell_states])
    n_O = sum([s == CELL_O for s in chunk_cell_states])
    n_X = sum([s == CELL_X for s in chunk_cell_states])
    if n_empty == len(chunk_cell_states):
        return chunk_obs_labels.index('Empty')
    if n_O > 0 and n_X > 0:
        return chunk_obs_labels.index('Unwinnable')
    if n_X == 0:
        return chunk_obs_labels.index('OIn1') + n_empty - 1
    if n_O == 0:
        return chunk_obs_labels.index('XIn1') + n_empty - 1
    raise ValueError("Invalid chunk state")

In [ ]:
# Cell observations
for i in range(n_cells):
    for state in range(3):
        A[i][state, state] = 1.

# Turn observation
for state in range(2):
    A[n_cells][state, state] = 1.0

# Empty count observation
for state in range(n_empty_count_states):
    A[n_cells+1][state, state] = 1.0

# Chunk observations.
# Each chunk needs to observe the state of its win_length component cells to see if it's empty, unwinnable, O in n moves, or X in n moves.
print("Cell states:", list(np.ndindex((3,) * win_length)))

for i, chunk in enumerate(chunks):
    for cell_states in np.ndindex((3,) * win_length):
        obs = get_chunk_obs(cell_states)
        print(f"Chunk {i} with cell states {cell_states} has obs: {obs} ({chunk_obs_labels[obs]})")
        A[n_cells+2+i][obs, *cell_states] = 1.0

print("A matrix dims:", [A[i].shape for i in range(len(A))])

Cell states: [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 1, 0), (0, 1, 1), (0, 1, 2), (0, 2, 0), (0, 2, 1), (0, 2, 2), (1, 0, 0), (1, 0, 1), (1, 0, 2), (1, 1, 0), (1, 1, 1), (1, 1, 2), (1, 2, 0), (1, 2, 1), (1, 2, 2), (2, 0, 0), (2, 0, 1), (2, 0, 2), (2, 1, 0), (2, 1, 1), (2, 1, 2), (2, 2, 0), (2, 2, 1), (2, 2, 2)]
Chunk 0 with cell states (0, 0, 0) has obs: 0 (Empty)
Chunk 0 with cell states (0, 0, 1) has obs: 4 (OIn2)
Chunk 0 with cell states (0, 0, 2) has obs: 7 (XIn2)
Chunk 0 with cell states (0, 1, 0) has obs: 4 (OIn2)
Chunk 0 with cell states (0, 1, 1) has obs: 3 (OIn1)
Chunk 0 with cell states (0, 1, 2) has obs: 1 (Unwinnable)
Chunk 0 with cell states (0, 2, 0) has obs: 7 (XIn2)
Chunk 0 with cell states (0, 2, 1) has obs: 1 (Unwinnable)
Chunk 0 with cell states (0, 2, 2) has obs: 6 (XIn1)
Chunk 0 with cell states (1, 0, 0) has obs: 4 (OIn2)
Chunk 0 with cell states (1, 0, 1) has obs: 3 (OIn1)
Chunk 0 with cell states (1, 0, 2) has obs: 1 (Unwinnable)
Chunk 0 with cell states (1, 1, 0)